In [2]:
import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html

# ── LOAD DATA ─────────────────────────────────────────────────────────────────
df = pd.read_excel(r"C:\Users\ODAMA\Downloads\06_NNPC_TimeSeries.xlsx",
                   sheet_name='Weekly Data',
                   header=2)

# ── KPIs ──────────────────────────────────────────────────────────────────────
total_revenue   = df['Total Revenue (₦)'].sum()
total_volume    = df['Volume Sold (Litres)'].sum()
avg_price       = df['Price per Litre (₦)'].mean()
total_stations  = df['Stations Supplied'].max()

# ── CHART DATA ────────────────────────────────────────────────────────────────

# Sort properly
month_order = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
df['Month'] = pd.Categorical(df['Month'], categories=month_order, ordered=True)
df = df.sort_values(['Year','Month','Week No.']).reset_index(drop=True)

# 1. Total Revenue by Month and Year (line)
monthly_rev = df.groupby(['Year','Month'])['Total Revenue (₦)'].sum().reset_index()
monthly_rev['Month-Year'] = monthly_rev['Month'].astype(str) + ' ' + monthly_rev['Year'].astype(str)

# 2. Volume Sold by Product (bar)
vol_by_product = df.groupby('Product')['Volume Sold (Litres)'].sum().reset_index().sort_values('Volume Sold (Litres)')

# 3. Revenue by Product (pie)
rev_by_product = df.groupby('Product')['Total Revenue (₦)'].sum().reset_index()

# 4. Price per Litre trend by Product (line)
price_trend = df.groupby(['Month','Year','Product'])['Price per Litre (₦)'].mean().reset_index()
price_trend['Month-Year'] = price_trend['Month'].astype(str) + ' ' + price_trend['Year'].astype(str)
price_trend = price_trend.sort_values(['Year','Month'])

# 5. Volume Sold by Year (grouped bar)
vol_by_year = df.groupby(['Year','Product'])['Volume Sold (Litres)'].sum().reset_index()

# 6. Stations Supplied over time
stations_trend = df.groupby(['Year','Month'])['Stations Supplied'].mean().reset_index()
stations_trend['Month-Year'] = stations_trend['Month'].astype(str) + ' ' + stations_trend['Year'].astype(str)
stations_trend = stations_trend.sort_values(['Year','Month'])

# ── CHART STYLE HELPER ────────────────────────────────────────────────────────
def style(fig):
    fig.update_layout(
        plot_bgcolor='#ffffff',
        paper_bgcolor='#ffffff',
        font=dict(color='#1f3b5c', family='Arial', size=13),
        title_font=dict(size=16, color='#1f3b5c', family='Arial'),
        margin=dict(l=20, r=20, t=50, b=100),
        height=400,
        showlegend=True,
        legend=dict(
            bgcolor='rgba(255,255,255,0.95)',
            bordercolor='#d6e6f2',
            borderwidth=1,
            font=dict(size=12, color='#1f3b5c'),
            orientation='h',
            yanchor='top',
            y=-0.25,
            xanchor='center',
            x=0.5
        ),
        xaxis=dict(tickangle=45, tickfont=dict(size=11), title_font=dict(size=13),
                   showgrid=True, gridcolor='#f0f4f8'),
        yaxis=dict(tickfont=dict(size=11), title_font=dict(size=13),
                   showgrid=True, gridcolor='#f0f4f8')
    )
    return fig

# ── CHARTS ────────────────────────────────────────────────────────────────────

# 1. Revenue Trend
fig_revenue = px.line(monthly_rev, x='Month-Year', y='Total Revenue (₦)',
                      markers=True, title='Total Revenue Trend by Month')
fig_revenue.update_traces(line_color='#1f6fbf', line_width=3,
                           marker=dict(color='#1f6fbf', size=8,
                                       line=dict(width=1.5, color='white')))
style(fig_revenue)

# 2. Volume by Product
fig_volume = px.bar(vol_by_product, x='Volume Sold (Litres)', y='Product',
                    orientation='h', color='Product',
                    color_discrete_sequence=['#08306b','#2171b5','#6baed6','#c6dbef'],
                    title='Total Volume Sold by Product')
style(fig_volume)

# 3. Revenue by Product — donut
fig_product_rev = px.pie(rev_by_product, names='Product', values='Total Revenue (₦)',
                          hole=0.5, title='Revenue Share by Product',
                          color_discrete_sequence=['#08306b','#2171b5','#6baed6','#c6dbef'])
style(fig_product_rev)

# 4. Price Trend by Product
fig_price = px.line(price_trend, x='Month-Year', y='Price per Litre (₦)',
                    color='Product', markers=True,
                    title='Price per Litre Trend by Product',
                    color_discrete_sequence=['#08306b','#2171b5','#6baed6','#c6dbef'])
fig_price.update_traces(line_width=2)
style(fig_price)

# 5. Volume by Year and Product
fig_year = px.bar(vol_by_year, x='Year', y='Volume Sold (Litres)',
                  color='Product', barmode='group',
                  color_discrete_sequence=['#08306b','#2171b5','#6baed6','#c6dbef'],
                  title='Volume Sold by Year and Product')
style(fig_year)

# 6. Stations Supplied Trend
fig_stations = px.area(stations_trend, x='Month-Year', y='Stations Supplied',
                       title='Avg Stations Supplied per Month',
                       color_discrete_sequence=['#2171b5'])
fig_stations.update_traces(line_color='#1f3b5c', line_width=3,
                             fillcolor='rgba(33,113,181,0.15)')
style(fig_stations)

# ── STYLES ────────────────────────────────────────────────────────────────────
BLUE_DARK  = '#1f3b5c'
BLUE_MID   = '#2171b5'
BLUE_LIGHT = '#d6e6f2'
WHITE      = '#ffffff'

kpi_card = {
    'backgroundColor': WHITE,
    'padding': '16px 22px',
    'borderRadius': '12px',
    'textAlign': 'center',
    'flex': '1',
    'margin': '6px',
    'border': f'1.5px solid {BLUE_LIGHT}',
    'boxShadow': '3px 3px 10px rgba(0,0,0,0.08)'
}
chart_box = {
    'backgroundColor': WHITE,
    'padding': '10px',
    'borderRadius': '12px',
    'boxShadow': '3px 3px 10px rgba(0,0,0,0.08)',
    'flex': '1',
    'border': f'1px solid {BLUE_LIGHT}'
}
sidebar_label = {
    'backgroundColor': BLUE_MID,
    'color': WHITE,
    'padding': '7px 12px',
    'borderRadius': '6px',
    'marginBottom': '7px',
    'fontSize': '13px',
    'fontWeight': 'bold',
    'textAlign': 'center'
}

# ── APP LAYOUT ────────────────────────────────────────────────────────────────
app = Dash(__name__)

app.layout = html.Div(
    style={'backgroundColor': '#eaf3f9', 'padding': '20px', 'fontFamily': 'Arial'},
    children=[

        # Title + KPI Row
        html.Div([
            html.Div(
                html.H2('NNPC — TIME SERIES ANALYSIS DASHBOARD',
                        style={'color': WHITE, 'margin': '0', 'fontSize': '20px'}),
                style={'backgroundColor': BLUE_DARK, 'padding': '16px 22px',
                       'borderRadius': '12px', 'flex': '2', 'marginRight': '12px'}
            ),
            html.Div([
                html.Div([
                    html.P('TOTAL REVENUE', style={'margin':'0 0 4px 0','fontSize':'12px','color':BLUE_MID,'fontWeight':'bold'}),
                    html.H3(f'₦{total_revenue/1e12:,.2f}T', style={'margin':'0','color':BLUE_DARK,'fontSize':'22px','fontWeight':'bold'})
                ], style=kpi_card),
                html.Div([
                    html.P('TOTAL VOLUME', style={'margin':'0 0 4px 0','fontSize':'12px','color':BLUE_MID,'fontWeight':'bold'}),
                    html.H3(f'{total_volume/1e9:,.2f}B L', style={'margin':'0','color':BLUE_DARK,'fontSize':'22px','fontWeight':'bold'})
                ], style=kpi_card),
                html.Div([
                    html.P('AVG PRICE/LITRE', style={'margin':'0 0 4px 0','fontSize':'12px','color':BLUE_MID,'fontWeight':'bold'}),
                    html.H3(f'₦{avg_price:,.0f}', style={'margin':'0','color':BLUE_DARK,'fontSize':'22px','fontWeight':'bold'})
                ], style=kpi_card),
                html.Div([
                    html.P('MAX STATIONS', style={'margin':'0 0 4px 0','fontSize':'12px','color':BLUE_MID,'fontWeight':'bold'}),
                    html.H3(f'{total_stations:,}', style={'margin':'0','color':BLUE_DARK,'fontSize':'22px','fontWeight':'bold'})
                ], style=kpi_card),
            ], style={'display': 'flex', 'flex': '4'})
        ], style={'display': 'flex', 'alignItems': 'center', 'marginBottom': '16px'}),

        # Body
        html.Div([

            # Sidebar
            html.Div([
                html.P('Products', style={'color':BLUE_MID,'fontWeight':'bold','marginBottom':'8px','fontSize':'13px'}),
                *[html.Div(p, style={**sidebar_label, 'fontSize': '11px'}) for p in ['PMS','AGO','DPK','LPG']],
                html.Br(),
                html.P('Years', style={'color':BLUE_MID,'fontWeight':'bold','marginBottom':'8px','fontSize':'13px'}),
                *[html.Div(str(y), style=sidebar_label) for y in [2022, 2023, 2024]],
                html.Br(),
                html.P('Metrics', style={'color':BLUE_MID,'fontWeight':'bold','marginBottom':'8px','fontSize':'13px'}),
                *[html.Div(m, style={**sidebar_label, 'fontSize': '11px'}) for m in ['Revenue','Volume','Price','Stations']],
                html.Br(), html.Br(),
                html.P('Prepared by', style={'fontSize':'12px','color':BLUE_DARK,'margin':'0'}),
                html.P('Odama Joseph', style={'fontSize':'13px','color':BLUE_DARK,'fontWeight':'bold','margin':'0'}),
            ], style={
                'width': '160px', 'minWidth': '160px',
                'backgroundColor': WHITE, 'padding': '16px',
                'borderRadius': '12px', 'boxShadow': '3px 3px 10px rgba(0,0,0,0.08)',
                'marginRight': '12px', 'border': f'1px solid {BLUE_LIGHT}'
            }),

            # Charts — 2 per row
            html.Div([
                html.Div([
                    html.Div(dcc.Graph(figure=fig_revenue,     config={'displayModeBar': False}), style=chart_box),
                    html.Div(dcc.Graph(figure=fig_volume,      config={'displayModeBar': False}), style=chart_box),
                ], style={'display': 'flex', 'gap': '12px', 'marginBottom': '12px'}),

                html.Div([
                    html.Div(dcc.Graph(figure=fig_product_rev, config={'displayModeBar': False}), style=chart_box),
                    html.Div(dcc.Graph(figure=fig_price,       config={'displayModeBar': False}), style=chart_box),
                ], style={'display': 'flex', 'gap': '12px', 'marginBottom': '12px'}),

                html.Div([
                    html.Div(dcc.Graph(figure=fig_year,     config={'displayModeBar': False}), style=chart_box),
                    html.Div(dcc.Graph(figure=fig_stations, config={'displayModeBar': False}), style=chart_box),
                ], style={'display': 'flex', 'gap': '12px'}),

            ], style={'flex': '1'})

        ], style={'display': 'flex', 'alignItems': 'flex-start'})
    ]
)

# ── RUN ───────────────────────────────────────────────────────────────────────
if __name__ == '__main__':
    app.run(debug=True, port=8056)

C:\Users\ODAMA\AppData\Local\Temp\ipykernel_15964\1607359629.py:24: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  monthly_rev = df.groupby(['Year','Month'])['Total Revenue (₦)'].sum().reset_index()
C:\Users\ODAMA\AppData\Local\Temp\ipykernel_15964\1607359629.py:34: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  price_trend = df.groupby(['Month','Year','Product'])['Price per Litre (₦)'].mean().reset_index()
C:\Users\ODAMA\AppData\Local\Temp\ipykernel_15964\1607359629.py:42: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False t